# Aggregated OneVsRest LR — validation-tuned thresholds (04-00b)

Standalone repair for the aggregated LR baseline of **04-00**.

The original 04-00 export (`01-results_multilabel/multilabel_results.csv`,
`AGG_LR_*` columns) tuned each drug's decision threshold directly on the
**test set**, unlike every other result in 01–04, which tunes on the
validation split. That makes the LR Balanced Accuracy optimistically biased
(mean 0.792 vs 0.782 validation-tuned; up to +0.053 on Amikacin).

This notebook re-runs only the LR part of the aggregated pipeline, exactly as
04-00/04-01 do:

- same union of the 10 per-drug files (4 sites, 26,642 samples),
- same species-stratified 70/15/15 split (seed 42),
- log1p + standardize fitted on the training split only,

but tunes the per-drug thresholds on the **validation** split and then scores
the untouched test set once.

Output: `01-results_multilabel/aggregated_lr_val_tuned.csv`
(drug, best C, threshold, val BalAcc/AUC, test BalAcc/AUC).

The notebook also uses — and rebuilds if corrupt — the shared data cache
`../04-01-MultiLabel-Cnn-Classifier/results_multilabel_cnn/aggregated_multilabel_data_cnn_v2.npz`,
so the next 04-01 run finds a valid cache.


In [ ]:
# =============================================================================
# 0. PATHS
# =============================================================================
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    print("Google Drive mounted")
    IN_COLAB = True
except ImportError:
    print("Running locally")
    IN_COLAB = False

if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
    DATA_ROOT = DRYAD / "Processed"
else:
    _CANDIDATES = [
        Path("/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed"),
        Path("/mnt/sda-home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed"),
        Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed"),
    ]
    DATA_ROOT = next((p for p in _CANDIDATES if p.exists()), _CANDIDATES[0])

# Run this notebook from its own directory (04-00-MultiLabel-CLassifier/)
HERE = Path.cwd()
OUT_DIR = HERE / "01-results_multilabel"
CACHE_PATH = (HERE.parent / "04-01-MultiLabel-Cnn-Classifier"
              / "results_multilabel_cnn" / "aggregated_multilabel_data_cnn_v2.npz")

if not DATA_ROOT.exists():
    print(f"WARNING: DATA_ROOT not found ({DATA_ROOT}); relying on the cache.")
assert OUT_DIR.exists(), f"results folder not found: {OUT_DIR} (run from the notebook directory)"

BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]
SITES = ["DRIAMS-A", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"OUT_DIR:   {OUT_DIR}")
print(f"CACHE:     {CACHE_PATH}")


In [ ]:
# =============================================================================
# 1. IMPORTS
# =============================================================================
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)


In [ ]:
# =============================================================================
# 2. MULTI-LABEL DATA LOADER (identical to 04-00/04-01)
# =============================================================================
def load_multilabel_split():
    """Build multi-label X/Y from Processed per-drug CSVs. A = train, B/C/D = test."""
    train_X, train_Y, train_sp = None, None, None
    test_sites = {}

    for site_name in SITES:
        proc_site = f"Proc_{site_name}"
        site_labels = {}; X_samples = {}; sp_samples = {}; site_drugs_ok = []

        for drug_name in DRUGS_10:
            p = DATA_ROOT / proc_site / drug_name.replace(" ", "_") / "data.csv"
            if not p.exists(): continue
            df = pd.read_csv(p)
            site_drugs_ok.append(drug_name)
            for _, row in df.iterrows():
                code = row["code"]
                if code not in X_samples:
                    X_samples[code] = row[BIN_COLS].values.astype("float32")
                    sp_samples[code] = row["species"]
                site_labels.setdefault(code, {})[drug_name] = row["label"]

        if not site_drugs_ok: continue
        codes = list(X_samples.keys())
        X = np.array([X_samples[c] for c in codes], dtype="float32")
        species = np.array([sp_samples[c] for c in codes])

        Y = np.full((len(codes), len(DRUGS_10)), np.nan, dtype=float)
        for i, drug in enumerate(DRUGS_10):
            for j, code in enumerate(codes):
                if drug in site_labels.get(code, {}):
                    Y[j, i] = site_labels[code][drug]

        n_lab = (~np.isnan(Y)).sum(axis=1).mean()
        print(f"  {site_name}: {len(codes)} samples, avg {n_lab:.1f} labels/sample")

        if site_name == "DRIAMS-A":
            train_X, train_Y, train_sp = X, Y, species
        else:
            test_sites[site_name] = (X, Y, species)

    print(f"\nTrain (A): {train_X.shape[0]} samples, {len(np.unique(train_sp))} species")
    for s, (Xs, Ys, sps) in test_sites.items():
        print(f"  Test {s}: {Xs.shape[0]} samples, {len(np.unique(sps))} species")
    return train_X, train_Y, train_sp, test_sites


In [ ]:
# =============================================================================
# 3. POOLED DATA (A+B+C+D) — shared cache with 04-01
# =============================================================================
def _load_cached_multilabel(path):
    """Return (X_all, Y_all, sp_all) from the npz cache, or None if unusable.

    npz members are read lazily; touching them forces decompression, so a
    truncated archive raises here instead of during the split.
    """
    try:
        with np.load(path, allow_pickle=True) as cached:
            X, Y, sp = cached["X_all"], cached["Y_all"], cached["sp_all"]
            assert X.shape[1] == 6000 and Y.shape == (X.shape[0], 10) and len(sp) == X.shape[0]
        return X, Y, sp
    except Exception as exc:  # BadZipFile (truncated), legacy/missing keys, ...
        print(f"Cache unusable ({type(exc).__name__}: {exc}) - rebuilding from CSVs")
        return None


def _save_cached_multilabel(path, X, Y, sp):
    """Atomic write: a crash mid-save cannot leave a truncated archive behind."""
    tmp = path.with_name(path.name + ".tmp.npz")
    np.savez_compressed(tmp, X_all=X, Y_all=Y, sp_all=sp)
    tmp.replace(path)
    print(f"Data saved to cache: {path}")


pooled = _load_cached_multilabel(CACHE_PATH) if CACHE_PATH.exists() else None
if pooled is None and not DATA_ROOT.exists():
    raise RuntimeError(
        f"Cache is missing/unusable and DATA_ROOT does not exist ({DATA_ROOT}).\n"
        "Restore the Processed dataset or copy a valid aggregate cache to "
        f"{CACHE_PATH}")
if pooled is None:
    print("Loading aggregated data from all 4 sites (first time)...")
    X_A, Y_A, sp_A, test_data_raw = load_multilabel_split()
    X_all = np.concatenate([X_A] + [v[0] for v in test_data_raw.values()])
    Y_all = np.concatenate([Y_A] + [v[1] for v in test_data_raw.values()])
    sp_all = np.concatenate([sp_A] + [v[2] for v in test_data_raw.values()])
    _save_cached_multilabel(CACHE_PATH, X_all, Y_all, sp_all)
else:
    X_all, Y_all, sp_all = pooled
    print(f"Loaded pooled data from cache: {CACHE_PATH}")

print(f"Total aggregated: {X_all.shape[0]} samples, {len(np.unique(sp_all))} species, "
      f"avg {(~np.isnan(Y_all)).sum(axis=1).mean():.2f} labels/sample")


In [ ]:
# =============================================================================
# 4. SPECIES-STRATIFIED 70/15/15 SPLIT (identical to 04-00/04-01, seed 42)
# =============================================================================
idx_trval, idx_test, _, _ = stratified_species_drug_split(
    np.arange(len(Y_all)).reshape(-1, 1), np.zeros(len(Y_all)),
    species=sp_all, test_size=0.15, random_state=SEED)
idx_trval = idx_trval.flatten().astype(int)
idx_test = idx_test.flatten().astype(int)

X_trval, Y_trval, sp_trval = X_all[idx_trval], Y_all[idx_trval], sp_all[idx_trval]
X_test, Y_test = X_all[idx_test], Y_all[idx_test]

val_frac = 0.15 / 0.85
idx_train, idx_val, _, _ = stratified_species_drug_split(
    np.arange(len(Y_trval)).reshape(-1, 1), np.zeros(len(Y_trval)),
    species=sp_trval, test_size=val_frac, random_state=SEED)
idx_train = idx_train.flatten().astype(int)
idx_val = idx_val.flatten().astype(int)

X_train, Y_train = X_trval[idx_train], Y_trval[idx_train]
X_val, Y_val = X_trval[idx_val], Y_trval[idx_val]
print(f"Split: train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

state = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state)
X_val_pp = apply_input_transform(X_val, state)
X_test_pp = apply_input_transform(X_test, state)
print("Preprocessing done (fit on train only).")


In [ ]:
# =============================================================================
# 5. OneVsRest LR — thresholds tuned on the VALIDATION split
# =============================================================================
C_grid = np.linspace(5e-5, 1e-3, 15)
thresholds = np.linspace(0.05, 0.95, 91)

rows = []
for di, drug in enumerate(tqdm(DRUGS_10, desc="AG LR (val-tuned)")):
    tr_mask = ~np.isnan(Y_train[:, di])
    X_tr, y_tr = X_train_pp[tr_mask], Y_train[tr_mask, di].astype(int)
    val_mask = ~np.isnan(Y_val[:, di])
    X_v, y_v = X_val_pp[val_mask], Y_val[val_mask, di].astype(int)
    te_mask = ~np.isnan(Y_test[:, di])
    X_te, y_te = X_test_pp[te_mask], Y_test[te_mask, di].astype(int)

    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                           max_iter=5000, random_state=SEED),
        param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy",
        n_jobs=-1, verbose=0)
    grid.fit(X_tr, y_tr)
    lr = grid.best_estimator_

    # Threshold: validation only
    proba_v = lr.predict_proba(X_v)[:, 1]
    best_t = thresholds[np.argmax(
        [balanced_accuracy_score(y_v, proba_v >= t) for t in thresholds])]

    val_balacc = balanced_accuracy_score(y_v, proba_v >= best_t)
    val_auc = roc_auc_score(y_v, proba_v)

    # Test: frozen threshold, scored once
    proba_te = lr.predict_proba(X_te)[:, 1]
    test_balacc = balanced_accuracy_score(y_te, proba_te >= best_t)
    test_auc = roc_auc_score(y_te, proba_te)

    rows.append({
        "Drug": drug, "Best_C": grid.best_params_["C"], "Threshold": best_t,
        "Val_BalAcc": val_balacc, "Val_AUC": val_auc,
        "Test_BalAcc": test_balacc, "Test_AUC": test_auc,
    })
    print(f"  {drug:35s} C={grid.best_params_['C']:.2e} t={best_t:.2f}  "
          f"val BA={val_balacc:.4f}  test BA={test_balacc:.4f}  test AUC={test_auc:.4f}")


In [ ]:
# =============================================================================
# 6. EXPORT + COMPARISON WITH THE ORIGINAL (TEST-TUNED) EXPORT
# =============================================================================
df = pd.DataFrame(rows)
out_path = OUT_DIR / "aggregated_lr_val_tuned.csv"
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}\n")
print(df.round(4).to_string(index=False))
print(f"\nMeans: Test BalAcc={df.Test_BalAcc.mean():.4f}  Test AUC={df.Test_AUC.mean():.4f}")

old_path = OUT_DIR / "multilabel_results.csv"
if old_path.exists():
    old = pd.read_csv(old_path)
    cmp = old[["Drug", "AGG_LR_BalAcc", "AGG_LR_AUC"]].merge(
        df[["Drug", "Test_BalAcc", "Test_AUC"]], on="Drug", how="left")
    cmp["dBallAcc"] = cmp["Test_BalAcc"] - cmp["AGG_LR_BalAcc"]
    cmp["dAUC"] = cmp["Test_AUC"] - cmp["AGG_LR_AUC"]
    print("\nOld (test-tuned) vs corrected (val-tuned):")
    print(cmp.round(4).to_string(index=False))
    print(f"\nOld mean BalAcc {cmp.AGG_LR_BalAcc.mean():.4f} -> "
          f"corrected {cmp.Test_BalAcc.mean():.4f}")
